# Pan-Cancer Biomarker Overlap Analysis

Overlap analysis restricted to **pan_cancer** hits across:
1. **Cohort**: cohort1 (first-line unmatched) vs cohort2 (1:1 line-matched)
2. **Weighting (Track 1)**: ATE (generalizability) vs unweighted
3. **Weighting (Track 2)**: ATE (IPTW) vs noIPTW
4. **PS model**: covariates_only vs covariates_plus_embeddings

For overlapping markers: HR direction concordance, effect size stability, and validation-level breakdown.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
import seaborn as sns

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 12,
    "legend.fontsize": 10,
    "figure.dpi": 150,
    "savefig.dpi": 200,
    "savefig.bbox": "tight",
})

# ── Paths ─────────────────────────────────────────────────────────────
DATA_PATH = '/data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/'
MARKER_PATH = os.path.join(DATA_PATH, 'biomarker_analysis/')
COMPILED_DIR = os.path.join(MARKER_PATH, 'compiled_results/')
FIGURE_PATH = '/data/gusev/USERS/jpconnor/figures/clinical_text_embedding_project/'
OVERLAP_FIG_PATH = os.path.join(FIGURE_PATH, 'biomarker_analysis/pan_cancer_overlap/')
os.makedirs(OVERLAP_FIG_PATH, exist_ok=True)

# ── Load compiled results (pan_cancer only) ───────────────────────────
t1_all = pd.read_csv(os.path.join(COMPILED_DIR, 'track1_all_significant_hits.csv'))
t2_all = pd.read_csv(os.path.join(COMPILED_DIR, 'track2_all_significant_hits.csv'))
findings = pd.read_csv(os.path.join(COMPILED_DIR, 'all_findings_with_validation.csv'))

t1 = t1_all[t1_all['cancer_type'] == 'pan_cancer'].copy()
t2 = t2_all[t2_all['cancer_type'] == 'pan_cancer'].copy()
findings_pan = findings[findings['cancer_type'] == 'pan_cancer'].copy()

print(f"Track 1 pan-cancer: {len(t1)} significant hits, {t1['marker'].nunique()} unique markers")
print(f"Track 2 pan-cancer: {len(t2)} significant hits, {t2['marker'].nunique()} unique markers")
print(f"All findings pan-cancer: {len(findings_pan)} rows")
print(f"\nTrack 1 specs present:")
for (c, p, w), g in t1.groupby(['cohort', 'ps_model', 'weight_type']):
    print(f"  {c} / {p} / {w}: {g['marker'].nunique()} markers")
print(f"\nTrack 2 specs present:")
for (c, p, w), g in t2.groupby(['cohort', 'ps_model', 'weight_type']):
    print(f"  {c} / {p} / {w}: {g['marker'].nunique()} markers")

In [ ]:
# ── Overlap utilities ─────────────────────────────────────────────────

def compute_pairwise_overlap(df, group_col, val_a, val_b, fixed_cols,
                             marker_col='marker', hr_col=None):
    """For each combination of fixed_cols, compute marker overlap between val_a and val_b."""
    rows = []
    df_a = df[df[group_col] == val_a]
    df_b = df[df[group_col] == val_b]

    combos_a = df_a[fixed_cols].drop_duplicates()
    combos_b = df_b[fixed_cols].drop_duplicates()
    combos = pd.merge(combos_a, combos_b, on=fixed_cols, how='outer')

    for _, combo in combos.iterrows():
        mask_a = pd.Series(True, index=df_a.index)
        mask_b = pd.Series(True, index=df_b.index)
        for col in fixed_cols:
            mask_a = mask_a & (df_a[col] == combo[col])
            mask_b = mask_b & (df_b[col] == combo[col])

        markers_a = set(df_a.loc[mask_a, marker_col])
        markers_b = set(df_b.loc[mask_b, marker_col])
        all_markers = markers_a | markers_b

        for m in sorted(all_markers):
            row = {c: combo[c] for c in fixed_cols}
            row['marker'] = m
            row[f'sig_{val_a}'] = m in markers_a
            row[f'sig_{val_b}'] = m in markers_b
            row['overlap'] = m in (markers_a & markers_b)

            if hr_col is not None:
                hr_a = df_a.loc[mask_a & (df_a[marker_col] == m), hr_col]
                hr_b = df_b.loc[mask_b & (df_b[marker_col] == m), hr_col]
                row[f'HR_{val_a}'] = float(hr_a.iloc[0]) if len(hr_a) else np.nan
                row[f'HR_{val_b}'] = float(hr_b.iloc[0]) if len(hr_b) else np.nan
                if not np.isnan(row[f'HR_{val_a}']) and not np.isnan(row[f'HR_{val_b}']):
                    row['direction_concordant'] = (
                        (row[f'HR_{val_a}'] > 1) == (row[f'HR_{val_b}'] > 1)
                    )
                else:
                    row['direction_concordant'] = np.nan
            rows.append(row)

    return pd.DataFrame(rows)


def print_overlap_summary(odf, val_a, val_b):
    n_total = len(odf)
    n_overlap = int(odf['overlap'].sum())
    n_a_only = int((odf[f'sig_{val_a}'] & ~odf[f'sig_{val_b}']).sum())
    n_b_only = int((odf[f'sig_{val_b}'] & ~odf[f'sig_{val_a}']).sum())

    print(f"  {val_a}-only: {n_a_only}  |  Both: {n_overlap}  |  {val_b}-only: {n_b_only}")
    if n_total > 0:
        print(f"  Jaccard: {n_overlap / n_total:.3f}")
    if 'direction_concordant' in odf.columns:
        conc = odf.loc[odf['overlap'], 'direction_concordant']
        if len(conc) > 0:
            nc = int(conc.sum())
            print(f"  HR direction concordance (overlap): {nc}/{len(conc)} "
                  f"({nc/len(conc)*100:.0f}%)")
    return n_a_only, n_overlap, n_b_only


print("Utilities defined.")

## 1. Track 1: Cohort Overlap (cohort1 vs cohort2)

In [ ]:
# ── Track 1: cohort1 vs cohort2 ───────────────────────────────────────

t1_cohort_ov = compute_pairwise_overlap(
    t1, group_col='cohort', val_a='cohort1', val_b='cohort2',
    fixed_cols=['ps_model', 'weight_type'],
    hr_col='HR_marker',
)

print("Track 1: cohort1 (unmatched) vs cohort2 (matched) — pan_cancer\n")

venn_counts_t1c = {}
for (ps, wt), grp in t1_cohort_ov.groupby(['ps_model', 'weight_type']):
    label = f"{ps} / {wt}"
    print(f"--- {label} ---")
    a, o, b = print_overlap_summary(grp, 'cohort1', 'cohort2')
    venn_counts_t1c[(ps, wt)] = (a, b, o)
    print()

# Venns
strata = sorted(venn_counts_t1c.keys())
fig, axes = plt.subplots(1, len(strata), figsize=(6 * len(strata), 5))
if len(strata) == 1:
    axes = [axes]
for i, key in enumerate(strata):
    venn2(subsets=venn_counts_t1c[key],
          set_labels=('cohort1\n(unmatched)', 'cohort2\n(matched)'), ax=axes[i])
    axes[i].set_title(f"{key[0]}\n{key[1]}", fontsize=11)
fig.suptitle("Track 1 Pan-Cancer: Matched vs Unmatched", fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(OVERLAP_FIG_PATH, 'T1_cohort_overlap_venns.png'))
plt.show()

## 2. Track 1: ATE vs Unweighted

In [ ]:
# ── Track 1: ATE vs unweighted ────────────────────────────────────────

t1_weight_ov = compute_pairwise_overlap(
    t1, group_col='weight_type', val_a='ATE', val_b='unweighted',
    fixed_cols=['cohort', 'ps_model'],
    hr_col='HR_marker',
)

print("Track 1: ATE vs unweighted — pan_cancer\n")

venn_counts_t1w = {}
for (coh, ps), grp in t1_weight_ov.groupby(['cohort', 'ps_model']):
    label = f"{coh} / {ps}"
    print(f"--- {label} ---")
    a, o, b = print_overlap_summary(grp, 'ATE', 'unweighted')
    venn_counts_t1w[(coh, ps)] = (a, b, o)
    print()

# Venns
strata = sorted(venn_counts_t1w.keys())
fig, axes = plt.subplots(1, len(strata), figsize=(6 * len(strata), 5))
if len(strata) == 1:
    axes = [axes]
for i, key in enumerate(strata):
    venn2(subsets=venn_counts_t1w[key],
          set_labels=('ATE', 'unweighted'), ax=axes[i])
    axes[i].set_title(f"{key[0]}\n{key[1]}", fontsize=11)
fig.suptitle("Track 1 Pan-Cancer: ATE vs Unweighted", fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(OVERLAP_FIG_PATH, 'T1_weight_overlap_venns.png'))
plt.show()

## 3. Track 1: PS Model Overlap (covariates_only vs covariates_plus_embeddings)

In [ ]:
# ── Track 1: covariates_only vs covariates_plus_embeddings ────────────

t1_ps_ov = compute_pairwise_overlap(
    t1, group_col='ps_model',
    val_a='covariates_only', val_b='covariates_plus_embeddings',
    fixed_cols=['cohort', 'weight_type'],
    hr_col='HR_marker',
)

print("Track 1: covariates_only vs covariates_plus_embeddings — pan_cancer\n")

venn_counts_t1ps = {}
for (coh, wt), grp in t1_ps_ov.groupby(['cohort', 'weight_type']):
    label = f"{coh} / {wt}"
    print(f"--- {label} ---")
    a, o, b = print_overlap_summary(grp, 'covariates_only', 'covariates_plus_embeddings')
    venn_counts_t1ps[(coh, wt)] = (a, b, o)
    print()

strata = sorted(venn_counts_t1ps.keys())
fig, axes = plt.subplots(1, len(strata), figsize=(6 * len(strata), 5))
if len(strata) == 1:
    axes = [axes]
for i, key in enumerate(strata):
    venn2(subsets=venn_counts_t1ps[key],
          set_labels=('cov_only', 'cov+emb'), ax=axes[i])
    axes[i].set_title(f"{key[0]} / {key[1]}", fontsize=11)
fig.suptitle("Track 1 Pan-Cancer: PS Model Overlap", fontweight='bold', y=1.02)
fig.tight_layout()
fig.savefig(os.path.join(OVERLAP_FIG_PATH, 'T1_ps_model_overlap_venns.png'))
plt.show()

## 4. Track 2: Cohort & Weighting Overlaps

In [ ]:
# ── Track 2: cohort overlap ───────────────────────────────────────────

t2_cohort_ov = compute_pairwise_overlap(
    t2, group_col='cohort', val_a='cohort1', val_b='cohort2',
    fixed_cols=['ps_model', 'weight_type'],
    hr_col='HR_markerxICI',
)

print("Track 2: cohort1 vs cohort2 — pan_cancer\n")
for (ps, wt), grp in t2_cohort_ov.groupby(['ps_model', 'weight_type']):
    print(f"--- {ps} / {wt} ---")
    print_overlap_summary(grp, 'cohort1', 'cohort2')
    if grp['overlap'].any():
        both = grp[grp['overlap']][['marker', f'HR_cohort1', f'HR_cohort2', 'direction_concordant']]
        display(both.reset_index(drop=True))
    print()

In [ ]:
# ── Track 2: ATE vs noIPTW ────────────────────────────────────────────

t2_weight_ov = compute_pairwise_overlap(
    t2, group_col='weight_type', val_a='ATE', val_b='noIPTW',
    fixed_cols=['cohort', 'ps_model'],
    hr_col='HR_markerxICI',
)

print("Track 2: ATE vs noIPTW — pan_cancer\n")
for (coh, ps), grp in t2_weight_ov.groupby(['cohort', 'ps_model']):
    print(f"--- {coh} / {ps} ---")
    print_overlap_summary(grp, 'ATE', 'noIPTW')
    if grp['overlap'].any():
        both = grp[grp['overlap']][['marker', 'HR_ATE', 'HR_noIPTW', 'direction_concordant']]
        display(both.reset_index(drop=True))
    print()

## 5. Cross-Specification Robustness Heatmap

Binary heatmap showing which pan-cancer markers survive across which specifications.
Rows = markers (sorted by number of specs), columns = (cohort, ps_model, weight_type).

In [ ]:
# ── Track 1: Cross-specification robustness heatmap ───────────────────

t1['spec'] = (t1['cohort'] + '\n' + t1['ps_model'].str.replace('covariates_', '') +
              '\n' + t1['weight_type'])

pivot_t1 = t1.pivot_table(index='marker', columns='spec', aggfunc='size', fill_value=0)
pivot_t1 = (pivot_t1 > 0).astype(int)
pivot_t1['n_specs'] = pivot_t1.sum(axis=1)
pivot_t1 = pivot_t1.sort_values('n_specs', ascending=False)
n_specs = pivot_t1.pop('n_specs')

fig_height = max(6, 0.35 * len(pivot_t1))
fig, ax = plt.subplots(figsize=(max(8, 1.2 * len(pivot_t1.columns)), fig_height))
sns.heatmap(pivot_t1, cmap=['#f0f0f0', '#2166ac'], cbar=False,
            linewidths=0.5, linecolor='white', ax=ax)
ax.set_title('Track 1 Pan-Cancer: Specification Robustness', fontweight='bold', pad=12)
ax.set_xlabel('Specification')
ax.set_ylabel('Marker')
ax.tick_params(axis='x', rotation=45)

# Annotate counts
for i, (marker, ns) in enumerate(n_specs.items()):
    ax.text(len(pivot_t1.columns) + 0.15, i + 0.5, f" {ns}/{len(pivot_t1.columns)}",
            va='center', fontsize=8, color='#333')

fig.tight_layout()
fig.savefig(os.path.join(OVERLAP_FIG_PATH, 'T1_robustness_heatmap.png'))
plt.show()

# List most robust markers
print(f"Markers in >= {len(pivot_t1.columns) - 1}/{len(pivot_t1.columns)} specifications:")
for marker in n_specs[n_specs >= len(pivot_t1.columns) - 1].index:
    print(f"  {marker} ({int(n_specs[marker])}/{len(pivot_t1.columns)})")

In [ ]:
# ── Track 2: Cross-specification robustness heatmap ───────────────────

if len(t2) > 0:
    t2['spec'] = (t2['cohort'] + '\n' + t2['ps_model'].str.replace('covariates_', '') +
                  '\n' + t2['weight_type'])

    pivot_t2 = t2.pivot_table(index='marker', columns='spec', aggfunc='size', fill_value=0)
    pivot_t2 = (pivot_t2 > 0).astype(int)
    pivot_t2['n_specs'] = pivot_t2.sum(axis=1)
    pivot_t2 = pivot_t2.sort_values('n_specs', ascending=False)
    n_specs_t2 = pivot_t2.pop('n_specs')

    fig_height = max(4, 0.5 * len(pivot_t2))
    fig, ax = plt.subplots(figsize=(max(8, 1.2 * len(pivot_t2.columns)), fig_height))
    sns.heatmap(pivot_t2, cmap=['#f0f0f0', '#d62728'], cbar=False,
                linewidths=0.5, linecolor='white', ax=ax, annot=True, fmt='d')
    ax.set_title('Track 2 Pan-Cancer: Specification Robustness', fontweight='bold', pad=12)
    ax.set_xlabel('Specification')
    ax.set_ylabel('Marker')
    ax.tick_params(axis='x', rotation=45)

    fig.tight_layout()
    fig.savefig(os.path.join(OVERLAP_FIG_PATH, 'T2_robustness_heatmap.png'))
    plt.show()

    print("Track 2 pan-cancer markers by robustness:")
    for marker in n_specs_t2.index:
        print(f"  {marker}: {int(n_specs_t2[marker])}/{len(pivot_t2.columns)} specs")
else:
    print("No Track 2 pan-cancer significant hits.")

## 6. HR Concordance Scatter Plots

For markers significant in both arms of each comparison, plot log2(HR) in one arm vs the other.

In [ ]:
# ── HR concordance scatter plots ──────────────────────────────────────

def plot_hr_concordance(odf, val_a, val_b, title, filename):
    """Scatter of log2(HR) in val_a vs val_b for overlapping markers."""
    both = odf[odf['overlap']].copy()
    if len(both) == 0:
        print(f"  No overlap for {title}")
        return

    hr_a_col = f'HR_{val_a}'
    hr_b_col = f'HR_{val_b}'
    both = both.dropna(subset=[hr_a_col, hr_b_col])
    both['logHR_a'] = np.log2(both[hr_a_col])
    both['logHR_b'] = np.log2(both[hr_b_col])

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(both['logHR_a'], both['logHR_b'],
               s=50, alpha=0.7, edgecolors='k', linewidth=0.5, c='#2166ac')

    # Label each point
    for _, row in both.iterrows():
        ax.annotate(row['marker'], (row['logHR_a'], row['logHR_b']),
                    fontsize=7, alpha=0.8,
                    xytext=(4, 4), textcoords='offset points')

    lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]),
            max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lims, lims, 'k--', alpha=0.3, lw=1, label='y=x')
    ax.axhline(0, color='grey', ls=':', lw=0.5)
    ax.axvline(0, color='grey', ls=':', lw=0.5)
    ax.set_xlim(lims)
    ax.set_ylim(lims)
    ax.set_xlabel(f'log2(HR) — {val_a}')
    ax.set_ylabel(f'log2(HR) — {val_b}')
    ax.set_title(title, fontweight='bold')

    n_conc = int(both['direction_concordant'].sum())
    n_tot = int(both['direction_concordant'].notna().sum())
    ax.text(0.98, 0.02,
            f"Concordance: {n_conc}/{n_tot} ({n_conc/max(n_tot,1)*100:.0f}%)",
            transform=ax.transAxes, ha='right', va='bottom', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    fig.tight_layout()
    fig.savefig(os.path.join(OVERLAP_FIG_PATH, filename))
    plt.show()


# Track 1: cohort overlap
plot_hr_concordance(
    t1_cohort_ov, 'cohort1', 'cohort2',
    'Track 1 Pan-Cancer: HR — Unmatched vs Matched',
    'T1_cohort_hr_concordance.png')

# Track 1: weight overlap
plot_hr_concordance(
    t1_weight_ov, 'ATE', 'unweighted',
    'Track 1 Pan-Cancer: HR — ATE vs Unweighted',
    'T1_weight_hr_concordance.png')

# Track 1: PS model overlap
plot_hr_concordance(
    t1_ps_ov, 'covariates_only', 'covariates_plus_embeddings',
    'Track 1 Pan-Cancer: HR — Cov-only vs Cov+Embeddings',
    'T1_ps_hr_concordance.png')

## 7. Validation-Level Breakdown for Overlap Markers

For markers that overlap across specifications, check their literature validation level.

In [ ]:
# ── Validation-level breakdown ────────────────────────────────────────

# Get unique markers that overlap on at least one axis
overlap_markers_set = set()
for odf in [t1_cohort_ov, t1_weight_ov, t1_ps_ov, t2_cohort_ov, t2_weight_ov]:
    if len(odf) > 0 and 'overlap' in odf.columns:
        overlap_markers_set.update(odf.loc[odf['overlap'], 'marker'])

print(f"Total unique pan-cancer markers overlapping on any axis: {len(overlap_markers_set)}\n")

# Merge with validation info (take first row per marker from findings)
val_info = (findings_pan[findings_pan['marker'].isin(overlap_markers_set)]
            .drop_duplicates(subset='marker', keep='first')
            [['marker', 'validation_level', 'validation_notes']]
            .sort_values('marker'))

LEVEL_ORDER = ['Very Strong', 'Strong', 'Moderate', 'Weak', 'Partial', 'Indirect', 'No Evidence']
val_info['validation_level'] = pd.Categorical(
    val_info['validation_level'], categories=LEVEL_ORDER, ordered=True)
val_info = val_info.sort_values('validation_level')

print("Validation level distribution:")
print(val_info['validation_level'].value_counts().reindex(LEVEL_ORDER).dropna().to_string())
print()
display(val_info.reset_index(drop=True))

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
counts = val_info['validation_level'].value_counts().reindex(LEVEL_ORDER).fillna(0)
colors_map = {
    'Very Strong': '#1a9850', 'Strong': '#66bd63', 'Moderate': '#a6d96a',
    'Weak': '#fee08b', 'Partial': '#fdae61', 'Indirect': '#f46d43',
    'No Evidence': '#d73027',
}
bars = ax.bar(counts.index, counts.values,
              color=[colors_map.get(l, '#999') for l in counts.index])
ax.set_ylabel('Count')
ax.set_title('Validation Level: Pan-Cancer Overlap Markers', fontweight='bold')
ax.tick_params(axis='x', rotation=30)
for bar, v in zip(bars, counts.values):
    if v > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                str(int(v)), ha='center', fontsize=10)

fig.tight_layout()
fig.savefig(os.path.join(OVERLAP_FIG_PATH, 'validation_level_overlap_markers.png'))
plt.show()

## 8. Export Overlap Markers for KM Analysis

In [ ]:
# ── Export for KM notebook ────────────────────────────────────────────

# Determine which track(s) each marker appeared in
t1_markers = set(t1['marker'].unique())
t2_markers = set(t2['marker'].unique())

km_rows = []
for m in sorted(overlap_markers_set):
    tracks = []
    if m in t1_markers:
        tracks.append(1)
    if m in t2_markers:
        tracks.append(2)
    km_rows.append({'marker': m, 'tracks': str(tracks)})

km_export = pd.DataFrame(km_rows)
km_export.to_csv(os.path.join(COMPILED_DIR, 'pan_cancer_overlap_markers_for_km.csv'), index=False)

print(f"Exported {len(km_export)} pan-cancer overlap markers for KM analysis:")
display(km_export)
print(f"\nSaved to {os.path.join(COMPILED_DIR, 'pan_cancer_overlap_markers_for_km.csv')}")